### OpenRouter API Key

This notebook uses OpenRouter through LangChain's OpenAI-compatible interface. Enter your OpenRouter API key when prompted. You do not need a separate OpenAI API key.


In [ ]:
# OPTIONAL: Install
# pip install -qU langchain langchain-openai langchain-community langchain-pinecone pinecone python-dotenv tiktoken


## Tutorial: Building a Full RAG Q&A with Pinecone 
We’ll ingest chunks into Pinecone, retrieve top‑k for a query, and answer with a grounded prompt.


In [ ]:
import os
from typing import List
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (hidden): ")

# Pinecone setup
INDEX_NAME = os.getenv("PINECONE_INDEX", "lc-demo-index")

from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
if INDEX_NAME not in [ix.name for ix in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region=os.getenv("PINECONE_REGION", "us-east-1"))
    )

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

llm = ChatOpenAI(model=MODEL, api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1", model=MODEL, temperature=0, seed=42)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)


### Step 1: Ingest documents into Pinecone
We’ll split raw text into chunks and upsert them with metadata.


In [ ]:
raw_corpus = (
    "LangChain is a framework for building LLM applications with prompts, chains, tools, and agents. "
    "It integrates with vector databases like Pinecone to enable retrieval‑augmented generation. "
    "Prompt templates, chunking, and retrievers are key to reliable Q&A."
)
chunks = text_splitter.split_text(raw_corpus)
metas = [{"source": "local", "chunk": i} for i in range(len(chunks))]
vectorstore.add_texts(texts=chunks, metadatas=metas)
print(f"Upserted {len(chunks)} chunks to index '{INDEX_NAME}'.")


### Step 2: Build a grounded Q&A prompt
We’ll keep it extractive and cite the `source` in metadata when helpful.


In [ ]:
qa_template = (
    "You are a precise assistant. Use the CONTEXT to answer the QUESTION.\n"
    "If not answerable from the CONTEXT, say: I don't know.\n\n"
    "CONTEXT:\n{context}\n\n"
    "QUESTION: {question}\n"
    "ANSWER:"
)
qa_prompt = PromptTemplate.from_template(qa_template)
qa_chain = LLMChain(llm=llm, prompt=qa_prompt)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def query_rag(question: str) -> str:
    docs = retriever.get_relevant_documents(question)
    context = "\n\n".join(d.page_content for d in docs)
    source = ", ".join(sorted({d.metadata.get("source", "unknown") for d in docs})) or "unknown"
    return qa_chain.run({"source": source, "context": context, "question": question})

print(query_rag("Name two LangChain building blocks."))


### Step 3: Ask a few questions
We’ll test multiple questions and observe grounded behavior.


In [ ]:
for q in [
    "Summarize what LangChain is in 1 line.",
    "What enables RAG with LangChain and Pinecone?",
    "What dataset does LangChain use to train?"
]:
    print("Q:", q)
    print(query_rag(q))
    print("-")
